# 02 — RRAO Capital Lines and Explain

This notebook computes the full capital result for the `rrao_sample_book_v1`
fixture under US NPR 2.0 and breaks it down into a line-level explain.

Key mechanics illustrated:
- The 10× risk-weight difference between EXOTIC (1%) and OTHER_RESIDUAL_RISK (0.1%)
- Which instruments drive the most capital per notional dollar
- How excluded positions appear in subtotals with zero add-on
- Investment-fund treatment: only the cited included-exposure portion is charged

Regulatory anchors: Basel MAR23.8; U.S. NPR 2.0 proposed section `__.211(c)`.

Prototype caution: outputs are synthetic fixture evidence.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.rrao_fixture import (
    load_context,
    load_positions,
    load_expected_outputs,
    PROFILE_US_NPR,
)
from frtb_rrao import calculate_rrao_capital, RraoClassification

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ctx = load_context()
positions = load_positions()
result = calculate_rrao_capital(positions, context=ctx)
expected = load_expected_outputs()

assert result.total_rrao == expected["total_rrao"], "Reconciliation check failed"

print(f"Total RRAO: USD {result.total_rrao:,.0f}")
print(f"Included lines: {len(result.lines)}")
print(f"Excluded lines: {len(result.excluded_lines)}")
print(f"Profile hash: {result.profile_hash[:16]}...")
print(f"Input hash: {result.input_hash[:16]}...")

## Line-Level Capital Explain

Every included position produces one deterministic `RraoCapitalLine`:
`add_on = gross_effective_notional × risk_weight`.  The table below sorts by
add-on descending so the largest capital drivers appear first.

The 10× risk-weight difference between EXOTIC (1%) and OTHER_RESIDUAL_RISK (0.1%)
makes EXOTIC classification the most capital-intensive outcome for a given notional.

In [ ]:
palette = {
    "EXOTIC": "#e05252",
    "OTHER_RESIDUAL_RISK": "#e0a040",
    "SUPERVISOR_DIRECTED": "#d96e28",
    "EXCLUDED": "#52a052",
}

sorted_lines = sorted(result.lines, key=lambda l: l.add_on, reverse=True)

rows = []
for line in sorted_lines:
    rows.append([
        line.position_id,
        line.classification.value,
        line.evidence_type.value,
        f"{line.risk_weight:.1%}",
        f"${line.gross_effective_notional/1e6:,.0f}M",
        f"${line.add_on:,.0f}",
    ])

display(Markdown(
    "| Position | Classification | Evidence Type | Risk Weight | Notional | Add-on (USD) |\n"
    "|---|---|---|---|---|---|\n"
    + "\n".join(f"| {' | '.join(r)} |" for r in rows)
))

print(f"\nTotal included add-on: ${sum(l.add_on for l in result.lines):,.0f}")

## Waterfall by Classification

RRAO capital is purely additive — there is no correlation offset, no netting,
and no diversification benefit across desks or evidence types.  This chart
shows the capital waterfall stacked by classification.

In [ ]:
from collections import defaultdict

cls_order = ["EXOTIC", "OTHER_RESIDUAL_RISK", "SUPERVISOR_DIRECTED"]
cls_addon: dict[str, float] = defaultdict(float)
cls_notional: dict[str, float] = defaultdict(float)
for line in result.lines:
    cls_addon[line.classification.value] += line.add_on
    cls_notional[line.classification.value] += line.gross_effective_notional

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Waterfall: add-on by classification
addons = [cls_addon.get(c, 0) for c in cls_order]
colors = [palette[c] for c in cls_order]
bars = ax1.bar(cls_order, addons, color=colors, alpha=0.86)
for bar, v in zip(bars, addons):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8000,
             f"${v:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.axhline(result.total_rrao, color="navy", linestyle="--", linewidth=1.4,
             label=f"Total RRAO ${result.total_rrao:,.0f}")
ax1.legend(fontsize=9)
ax1.set_title("Capital add-on by classification")
ax1.set_ylabel("USD add-on")
ax1.set_xticklabels([c.replace("_", "\n") for c in cls_order])

# Capital intensity: add-on per $1M of notional
notionals_m = [cls_notional.get(c, 0) / 1e6 for c in cls_order]
intensity = [a / n if n > 0 else 0 for a, n in zip(addons, notionals_m)]
bars2 = ax2.bar(cls_order, intensity, color=colors, alpha=0.86)
for bar, v in zip(bars2, intensity):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
             f"${v:,.0f}/M", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.set_title("Capital intensity (USD add-on per $1M notional)")
ax2.set_ylabel("USD per $1M notional")
ax2.set_xticklabels([c.replace("_", "\n") for c in cls_order])

fig.suptitle("RRAO Capital Waterfall — US NPR 2.0", fontsize=13)
plt.tight_layout()
plt.show()

print("Capital intensity: EXOTIC is 10× OTHER_RESIDUAL_RISK by construction (1% vs 0.1%)")

## Top Capital Drivers

This chart ranks all included positions by add-on and colours them by
classification.  Behavioural risk positions dominate by notional size
($800M combined) but the 0.1% weight keeps their capital contribution in
line with other OTHER_RESIDUAL_RISK positions.  Fund Alpha's exotic
portion produces $300k from only $30M included notional.

In [ ]:
sorted_included = sorted(result.lines, key=lambda l: l.add_on, reverse=True)
names = [l.position_id.replace("-001", "") for l in sorted_included]
addons_arr = np.array([l.add_on for l in sorted_included])
colors_arr = [palette[l.classification.value] for l in sorted_included]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(names[::-1], addons_arr[::-1], color=colors_arr[::-1], alpha=0.86)
for bar, v in zip(bars, addons_arr[::-1]):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height() / 2,
            f"${v:,.0f}", va="center", fontsize=8)

import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=palette[c], label=c) for c in ["EXOTIC", "OTHER_RESIDUAL_RISK", "SUPERVISOR_DIRECTED"]]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8)
ax.set_title("Capital add-on by position (USD) — US NPR 2.0")
ax.set_xlabel("Add-on (USD)")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## Investment Fund Line Detail

US NPR 2.0 section `__.211(a)(3)` includes a portion of investment-fund
exposures in RRAO.  The charged notional is the fund gross effective notional
multiplied by the cited `included_exposure_ratio`; the full fund notional is
**not** charged — only the portion that meets the mandate evidence requirement.

Fund Alpha carries 15% of its mandate as exotic exposure → charged at 1%.
Fund Beta carries 25% as other-residual-risk exposure → charged at 0.1%.

In [ ]:
fund_lines = [l for l in result.lines if "fund" in l.position_id]
fund_positions = {p.position_id: p for p in positions if "fund" in p.position_id}

print("Investment-fund RRAO lines (US NPR __.211(a)(3)):")
print()
for line in fund_lines:
    pos = fund_positions[line.position_id]
    desc = pos.investment_fund_descriptor
    print(f"Position: {line.position_id}")
    print(f"  Fund ID:               {desc.fund_id}")
    print(f"  Fund gross notional:   ${desc.fund_gross_effective_notional/1e6:,.0f}M")
    print(f"  Included ratio:        {desc.included_exposure_ratio:.0%}")
    print(f"  Charged notional:      ${line.gross_effective_notional/1e6:,.0f}M  (= fund × ratio)")
    print(f"  Exposure type:         {desc.included_exposure_type.value}")
    print(f"  Risk weight:           {line.risk_weight:.1%}")
    print(f"  Add-on:                ${line.add_on:,.0f}")
    print(f"  Citations:             {', '.join(line.citations)}")
    print()

## Excluded Lines: Gross Notional at Zero Cost

Excluded positions do not contribute to the RRAO add-on, but they do appear
in the result's subtotals with zero add-on.  This makes it possible to audit
the *scope* of the exclusion (how much gross notional was zeroed out) separately
from the capital computation.

In [ ]:
excl_notional = sum(l.gross_effective_notional for l in result.excluded_lines)
incl_notional = sum(l.gross_effective_notional for l in result.lines)

print(f"Included notional:   ${incl_notional/1e6:>8,.0f}M  → RRAO ${result.total_rrao:,.0f}")
print(f"Excluded notional:   ${excl_notional/1e6:>8,.0f}M  → RRAO $0")
print(f"Total gross notional: ${(incl_notional + excl_notional)/1e6:>7,.0f}M")
print()

excl_order = sorted(result.excluded_lines, key=lambda l: l.gross_effective_notional, reverse=True)
excl_names = [l.position_id.replace("-001", "") for l in excl_order]
excl_notionals = [l.gross_effective_notional / 1e6 for l in excl_order]
excl_reasons = [l.exclusion_reason.value if l.exclusion_reason else "?" for l in excl_order]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(excl_names[::-1], excl_notionals[::-1], color="#52a052", alpha=0.75)
for bar, n, reason in zip(bars, excl_notionals[::-1], excl_reasons[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"${n:,.0f}M  ({reason})", va="center", fontsize=8)
ax.set_title("Excluded position gross notional (add-on = $0) — US NPR 2.0")
ax.set_xlabel("USD millions")
plt.tight_layout()
plt.show()